In [ ]:
#%matplotlib inline
%time from hikyuu.interactive import *

# 1 利用 TM 实现简单的记账本

TradeManager对象可以理解为一个模拟的交易账户，负责交易的买/卖操作、记录交易记录以及持仓情况，也可以通过修改其买/卖操作的接口实现实盘接入。创建一个模拟交易账户，通常使用快捷创建函数 crtTM。TM对象的基本操作：

- buy  买入
- sell 卖出
- checkin 存入现金
- checkout 取出现金

可以利用 TM 实现简单的记账本，手工记录自己的操作情况，例如：

In [ ]:
# Create a simulated account with an initial capital of 100,000, starting from 2017-01-01
my_tm = crtTM(init_cash=100000, date=Datetime(201701010000))

# On 2017-01-03, buy 100 shares at the price of 9.11
td = my_tm.buy(Datetime(201701030000), sm['sz000001'], 9.11, 100)

# View the current cash and position
print(my_tm)

In [ ]:
# Convert to a pandas DataFrame to display the current positions 
position = my_tm.get_position_list()
position.to_df()

In [ ]:
my_tm.get_trade_list().to_df()

In [ ]:
# On 2017-02-21, sell 100 shares at the price of 9.60
td = my_tm.sell(Datetime(201702210000), sm['sz000001'], 9.60)

my_tm

# 2 利用 Excel 查看交易详情

使用 tocsv 方法将 TM 的交易记录、当前持仓及已平仓详细情况分别保存为 csv 文件，以便用 Excel 查看详情。

tocsv方法参数为一个指定的目录，目录必须以存在。其输出会在指定目录中，生成三个文件，“TM名称_交易记录.csv”、“TM名称_未平仓记录.csv”、“TM名称_已平仓记录.csv”。TM名称可在crtTM创建TM对象时指定，默认为“SYS”，如下图所示。

<img src="images/008_01_tocsv.png" align='left'>

In [ ]:
# Output to the temporary path configured in the hikyuu_XXX.ini file
my_tm.tocsv(sm.tmpdir())

使用 Excel 查看 csv，如：

<img src="images/008_02_tocsv_look.png" align="left">

# 3 使用序列化保存或重新载入已有TM对象

In [ ]:
# Save to the specified file
from datetime import date
filename = "my_trade_record_{}.pkl".format(date.today())
hku_save(my_tm, filename)

In [ ]:
# Load the saved TM object
new_my_tm = hku_load(filename)

# 4 使用订单代理

In [ ]:
# Create a simulated trading account for backtesting with an initial capital of 300,000
my_tm = crtTM(init_cash=300000, date=Datetime(201701010000))

# Register the live trading order broker
ob = crtOB(TestOrderBroker())
my_tm.reg_broker(ob) # TestOerderBroker is a test order broker object that only prints
# Note: pybind does not support the following calling style; you must create the instance first and then pass it!!!
# my_tm.reg_broker(crtOB(TestOrderBroker(), False))

# Modify the last datetime of the order broker as needed; only after this datetime will the order broker actually issue the order instructions
my_tm.broker_last_datetime=Datetime(201701010000)

# Create the signal generator (the 5-day EMA as the fast line and the 10-day EMA of the 5-day EMA itself as the slow line; buy when the fast line crosses the slow line upward, and sell otherwise)
my_sg = SG_Flex(EMA(C, n=5), slow_n=10)

# Fixedly buy 1000 shares each time
my_mm = MM_FixedCount(1000)

# Create the trading system and run it
sys = SYS_Simple(tm = my_tm, sg = my_sg, mm = my_mm)
sys.run(sm['sz000001'], Query(-150))

In [ ]:
my_tm.get_trade_list().to_df()

In [ ]:
my_tm.get_trade_list().to_np()

In [ ]:
my_tm.get_history_position_list().to_df()

In [ ]:
my_tm.get_history_position_list().to_np()